In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Load the Iris Dataset
# We use the built-in sklearn dataset which is standard for this task
data = load_iris()
X = data.data
y = data.target
target_names = data.target_names

# 2. Split the data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# 3. Standardize the Features (IMPORTANT for PCA)
# PCA is sensitive to scale, so we must normalize the data first
sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std = sc.transform(X_test)

# 4. Apply PCA
# Reducing 4 dimensions down to 2 for easy visualization
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

# Check how much information we retained
explained_variance = pca.explained_variance_ratio_
total_var = sum(explained_variance) * 100
print(f"Explained Variance by Component: {explained_variance}")
print(f"Total Information Retained: {total_var:.2f}%")

# 5. Train Classifier (Random Forest)
# We train on the REDUCED data (2 columns instead of 4)
classifier = RandomForestClassifier(max_depth=2, random_state=0)
classifier.fit(X_train_pca, y_train)

# 6. Evaluate Performance
y_pred = classifier.predict(X_test_pca)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("-" * 30)
print(f"Model Accuracy (using PCA data): {acc * 100:.2f}%")
print("Confusion Matrix:\n", cm)
print("-" * 30)

# 7. Visualize the PCA Result
plt.figure(figsize=(10, 6))
colors = ['red', 'green', 'blue']

# Plot each class with a different color
for i, target_name in enumerate(target_names):
    plt.scatter(X_train_pca[y_train == i, 0], X_train_pca[y_train == i, 1], 
                color=colors[i], label=target_name, edgecolor='k', s=100, alpha=0.7)

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title(f'PCA of Iris Dataset (Retained {total_var:.1f}% Variance)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('pca_result.png')
plt.show()